# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## Ruta de esta práctica · ML · Encuentro 1: regresión
Recta → curva → complejidad y regularización. Los tres bloques generan datos distintos; solo OLS/Ridge/Lasso del tercer bloque comparten prueba.

**Antes de comenzar:** guarda tu copia personal. Usa el índice lateral de Colab para localizar el bloque indicado. Ejecuta las celdas de preparación y después el bloque, en orden; cambiar una variable exige volver a ejecutar su celda y las que dependen de ella. Abrir una figura no vuelve a entrenar un modelo.

**Entorno:** estos ejemplos están preparados para CPU. No requieren claves de API, contratar servicios ni activar un entorno de pago. La disponibilidad y límites de los recursos de Colab dependen de tu cuenta; no cambies a recursos de pago para esta práctica. Si se interrumpe la sesión, conserva el cuaderno y vuelve a ejecutar desde la preparación.

**Distingue siempre:** datos observados o simulados → parámetros aprendidos por `fit`/entrenamiento → configuraciones que tú eliges → resultados en datos reservados. Una buena cifra de entrenamiento no demuestra aprendizaje útil en nuevos casos.


## 1. Una recta para datos de fertilizante
Datos simulados: X es fertilizante (kg/ha), y es rendimiento (t/ha). `fit` aprende la recta; `predict` estima valores. Los puntos son observaciones y la línea es el ajuste. Esta primera figura usa entrenamiento; todavía no mide generalización. La pendiente describe asociación en el modelo, no una dosis causal recomendada.

### Entender antes de ejecutar: pendiente, intercepto y predicción
La recta calcula **ŷ = b + w·x**. `fit` elige pendiente w e intercepto b minimizando errores cuadrados en las diez parejas inventadas; no los escribimos como respuesta. Con x=100 kg/ha, la próxima celda adicional sustituye los valores aprendidos y muestra la operación.

En el gráfico, cada punto es una pareja observada y la línea es una predicción. La separación vertical es un error en t/ha. El intercepto corresponde a x=0, fuera del rango 50–140: no lo interpretes como rendimiento comprobado sin fertilizante. Cambiar x al predecir usa la recta; cambiar las parejas y ejecutar `fit` la vuelve a aprender.

In [ ]:
# ============================================
# Ejemplo simple de Regresión Lineal en Google Colab
# Contexto: Agronegocios
# ============================================

# -----------------------
# 1) IMPORTAR LIBRERÍAS
# -----------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# -----------------------
# 2) CREAR DATOS DE EJEMPLO
# -----------------------
# Relación: "Cantidad de fertilizante (kg/ha)" → "Rendimiento de maíz (ton/ha)"
X = np.array([50, 60, 70, 80, 90, 100, 110, 120, 130, 140]).reshape(-1, 1)
y = np.array([2.1, 2.4, 2.8, 3.1, 3.5, 3.7, 4.0, 4.3, 4.6, 4.8])

# -----------------------
# 3) AJUSTAR MODELO LINEAL
# -----------------------
modelo = LinearRegression()
modelo.fit(X, y)

# Predicciones
y_pred = modelo.predict(X)

# -----------------------
# 4) VISUALIZACIÓN
# -----------------------
plt.scatter(X, y, color="green", label="Observaciones simuladas (maíz)")
plt.plot(X, y_pred, color="red", linewidth=2, label="Regresión lineal")
plt.xlabel("Fertilizante aplicado (kg/ha)")
plt.ylabel("Rendimiento (ton/ha)")
plt.title("Regresión Lineal: Fertilizante vs Rendimiento de Maíz")
plt.legend()
plt.show()

# -----------------------
# 5) RESULTADOS
#Datos: construimos diez observaciones hipotéticas; no son mediciones de campo ni un ensayo de fertilización.
#Relación: en esta tabla inventada, los valores mayores de fertilizante se asocian con rendimientos mayores.
#Modelo: ajusta una recta que predice la producción esperada de maíz dado un nivel de fertilización.
#Interpretación: la pendiente expresa cuánto cambia la predicción en t/ha por +1 kg/ha de fertilizante; no estima un efecto causal.
# -----------------------

print("Pendiente (coeficiente):", modelo.coef_[0])
print("Intercepto:", modelo.intercept_)

### Sustituye los parámetros aprendidos
Esta operación usa la recta anterior sin volver a ajustarla. Compara el cálculo manual con `predict`.

In [ ]:
x_consulta = 100.0
aporte = float(modelo.coef_[0]) * x_consulta
manual = float(modelo.intercept_) + aporte
calculada = float(modelo.predict([[x_consulta]])[0])
print(f'ŷ = {modelo.intercept_:.6f} + {modelo.coef_[0]:.6f} × {x_consulta:.0f} = {manual:.6f} t/ha')
assert np.isclose(manual, calculada)


## 2. Una curva para una relación diferente
Este bloque cambia los datos: no es una comparación con la recta anterior sobre el mismo conjunto. El polinomio añade potencias de X. Lee la curva dentro del rango observado; un máximo del ejemplo no constituye una recomendación agronómica.

### Qué cambia al añadir potencias
Con grado 2 usamos 1, x y x². El modelo sigue siendo lineal en sus coeficientes, aunque su gráfico frente a x sea curvo. Los coeficientes se aprenden; el grado lo elegimos. No confundas x² con duplicar x: para x=100, x²=10.000. La regla generadora y los datos son diferentes al primer bloque; una curva no ha «ganado» a esa recta por comparación justa.

In [ ]:
# ============================================
# Ejemplo de Regresión Polinómica en Google Colab
# Contexto: Agronegocios
# ============================================

# -----------------------
# 1) IMPORTAR LIBRERÍAS
# -----------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# -----------------------
# 2) CREAR DATOS DE EJEMPLO
# -----------------------
# Relación: "Cantidad de fertilizante (kg/ha)" → "Rendimiento de maíz (ton/ha)"
# En esta tabla construida, los rendimientos bajan en el tramo de valores altos de X; esa forma fue elegida para la simulación.
X = np.array([50, 60, 70, 80, 90, 100, 110, 120, 130, 140]).reshape(-1, 1)
y = np.array([2.1, 2.7, 3.5, 4.2, 4.8, 5.0, 4.9, 4.6, 4.0, 3.5])

# -----------------------
# 3) TRANSFORMAR VARIABLES (POLINOMIO)
# -----------------------
poly = PolynomialFeatures(degree=2)  # cuadrático
X_poly = poly.fit_transform(X)

# Ajustar modelo polinómico
modelo_poly = LinearRegression()
modelo_poly.fit(X_poly, y)

# Predicciones
y_pred = modelo_poly.predict(X_poly)

# -----------------------
# 4) VISUALIZACIÓN
# -----------------------
plt.scatter(X, y, color="green", label="Observaciones simuladas (maíz)")
plt.plot(X, y_pred, color="red", linewidth=2, label="Regresión polinómica (grado 2)")
plt.xlabel("Fertilizante aplicado (kg/ha)")
plt.ylabel("Rendimiento (ton/ha)")
plt.title("Regresión Polinómica: Fertilizante vs Rendimiento de Maíz")
plt.legend()
plt.show()

# -----------------------
# 5) RESULTADOS
# Contexto: estos datos inventados permiten estudiar una curva; no demuestran el efecto de aplicar fertilizante a un cultivo.
# Polinómica: la curva (parábola) captura esa relación no lineal: crecimiento inicial, máximo, y caída posterior.
# Interpretación: el máximo pertenece a la curva ajustada del ejemplo y al rango observado; no es una dosis agronómica recomendada.
# -----------------------
print("Coeficientes:", modelo_poly.coef_)
print("Intercepto:", modelo_poly.intercept_)


## 3. Controlar complejidad con Ridge y Lasso
Ahora usamos fertilizante y lluvia. La misma partición permite comparar OLS, Ridge y Lasso. Mira RMSE en entrenamiento y prueba; la regularización no gana por definición. En la curva parcial la lluvia se mantiene fija, mientras los puntos tienen lluvias diferentes.

### Qué controlan grado y alpha
Grado 5 con dos variables crea 20 términos sin contar el intercepto: incluye x₁, x₂, x₁² y x₁x₂, entre otros. Escalamos esos términos dentro del pipeline, usando solo entrenamiento. Ridge añade una penalización proporcional a la suma de coeficientes cuadrados (L2); Lasso a la suma de valores absolutos (L1). **alpha** es la fuerza elegida de la penalización; no es un coeficiente aprendido ni una tasa de aprendizaje.

RMSE = √[promedio de (real−predicho)²], en t/ha. Para errores 0,2 y −0,4: √[(0,04+0,16)/2]≈0,316. La tabla separa entrenamiento y prueba. Elegir grado o alpha por sus resultados de prueba convertiría esa prueba en validación.

In [ ]:
# ============================================================
# Regresión con Regularización: Ridge y Lasso (evitar sobreajuste)
# Contexto: Agronegocios
# Objetivo: predecir rendimiento de maíz (ton/ha) usando fertilizante y clima.
# ============================================================

# 1) IMPORTS
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 2) DATOS (sintéticos; rangos y coeficientes elegidos para el ejemplo)
# Variables base:
#   x1 = fertilizante (kg/ha)
#   x2 = lluvia en ciclo (mm)
# Intención: relación no lineal suave + ruido -> ideal para mostrar sobreajuste.
rng = np.random.RandomState(42)
n = 60
x1 = rng.uniform(60, 180, size=n)                 # fertilizante
x2 = rng.uniform(200, 800, size=n)                # lluvia
# Regla generadora inventada (desconocida para el modelo):
y_true = (
    0.02*x1 + 0.004*x2
    - 0.00006*(x1**2)                             # término cuadrático elegido para la simulación
    + 0.0000005*(x2**2)                           # término cuadrático elegido para la simulación
    + 0.00003*x1*x2                               # interacción
)
y = y_true + rng.normal(0, 0.3, size=n)           # ruido
X = np.c_[x1, x2]

# 3) TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

# 4) DEFINICIÓN DE PIPELINES
# PolynomialFeatures genera 20 términos con dos entradas y grado máximo 5.
# Aumentar flexibilidad puede favorecer el sobreajuste; no lo vuelve inevitable.
poly_degree = 5

ols = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
    ("scaler", StandardScaler(with_mean=False)),  # escala términos; el modelo ajusta su intercepto
    ("linreg", LinearRegression())
])

ridge = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
    ("scaler", StandardScaler(with_mean=False)),
    ("ridge", Ridge(alpha=5.0, random_state=0))   # alpha controla la fuerza de L2
])

lasso = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
    ("scaler", StandardScaler(with_mean=False)),
    ("lasso", Lasso(alpha=0.02, max_iter=50000, random_state=0))  # L1 puede poner coef=0
])

# 5) ENTRENAR
ols.fit(X_train, y_train)
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

# 6) EVALUACIÓN
def eval_model(name, pipe):
    yhat_tr = pipe.predict(X_train)
    yhat_te = pipe.predict(X_test)
    rmse_train = mean_squared_error(y_train, yhat_tr) ** 0.5
    rmse_test = mean_squared_error(y_test, yhat_te) ** 0.5
    return {
        "name": name,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "r2_train": r2_score(y_train, yhat_tr),
        "r2_test": r2_score(y_test, yhat_te),
        "n_coefs_nonzero": int(np.sum(
            np.abs(pipe.named_steps[list(pipe.named_steps.keys())[-1]].coef_) > 1e-8
        ))
    }
results = [
    eval_model("OLS (sin regularización)", ols),
    eval_model("Ridge (L2)", ridge),
    eval_model("Lasso (L1)", lasso),
]

for r in results:
    print(r["name"])
    print(f"  RMSE train: {r['rmse_train']:.3f} | RMSE test: {r['rmse_test']:.3f}")
    print(f"  R2   train: {r['r2_train']:.3f} | R2   test: {r['r2_test']:.3f}")
    print(f"  #coeficientes ≠ 0: {r['n_coefs_nonzero']}")
    print("-"*60)

# 7) VISUAL: curva parcial vs fertilizante manteniendo lluvia en su mediana
fert_grid = np.linspace(X_train[:,0].min(), X_train[:,0].max(), 120)
lluvia_med = np.median(X_train[:,1])
X_plot = np.c_[fert_grid, np.full_like(fert_grid, lluvia_med)]

plt.figure(figsize=(6,4))
plt.scatter(X_train[:,0], y_train, s=18, alpha=0.6, label="Train")
plt.scatter(X_test[:,0],  y_test,  s=18, alpha=0.6, label="Test")
plt.plot(fert_grid, ols.predict(X_plot),   lw=2, label="OLS")
plt.plot(fert_grid, ridge.predict(X_plot), lw=2, label="Ridge")
plt.plot(fert_grid, lasso.predict(X_plot), lw=2, label="Lasso")
plt.xlabel("Fertilizante (kg/ha)")
plt.ylabel("Rendimiento estimado (ton/ha)")
plt.title(f"Modelo parcial vs fertilizante (lluvia={lluvia_med:.0f} mm)")
plt.legend()
plt.tight_layout()
plt.show()

# 8) RESULTADOS (explicación práctica)
print("\nInterpretación práctica:")
print("""
- OLS ajusta los 20 términos por mínimos cuadrados, sin penalización. Una brecha
  entre entrenamiento y prueba puede advertir sobreajuste; consulta sus métricas.
- Ridge (L2) penaliza la magnitud de los coeficientes. Puede reducir varianza,
  a cambio de introducir sesgo; no garantiza mejorar en cualquier partición.
- Lasso (L1) puede dejar coeficientes exactamente en 0: aquí selecciona términos
  polinómicos, no necesariamente variables originales completas.
- Lee la tabla para ver qué modelos redujeron el RMSE de prueba con tu configuración.
  Para elegir grado y alpha, usaríamos validación dentro del entrenamiento, sin ajustar con TEST.
- La curva parcial mantiene fija la lluvia en la mediana de entrenamiento. Describe
  predicciones de esta simulación; no valida efectos causales ni dosis recomendadas.
""")
